# Overlay Histograms - vessel diameter

Generate histogram of the vessel diameter for the CD105 vessel system

Input: circular objects found in the CD105 segmentation in the "06_CircularObjects.ijm" Fiji script

Output: Histogram of vessel diameter, for vessel either runing along the central vein or perpendicular to the central vein

In [1]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd

## Define standard colors and legend titles

In [2]:
# Young mice without blood loss => shades of blue
Ycolors = ['lightblue', 'deepskyblue', 'dodgerblue', 'blue', 'navy']
Ylabels = ['VK-AA498', 'VK-AA499', 'VK-AA500', 'VK-AA501', 'VK-AA502']

# Young. blood loss mice => shades of orange
Bcolors = ['peachpuff', 'sandybrown', 'darkorange', 'chocolate', 'saddlebrown']
Blabels = ['VK-AA763', 'VK-AA764', 'VK-AA765', 'VK-AA766', 'VK-AA767']

# Old mice => shades of grey
Ocolors = ['black', 'dimgrey', 'grey', 'darkgrey', 'gainsboro']
Olabels = ['VK-AA758', 'VK-AA759', 'VK-AA760', 'VK-AA761', 'VK-AA762']

## Define datasets

In [3]:
# Young mice (both x-y and z-y are perpendicular to the central vein)
Ymouse_1_xy = 'VK-AA498_shaft_2376-3055_x-y.csv'
Ymouse_2_xy = 'VK-AA499_shaft_2065-2655_x-y.csv'
Ymouse_3_xy = 'VK-AA500_shaft_2506-3222_x-y.csv'
Ymouse_4_xy = 'VK-AA501_shaft_2376-3055_x-y.csv'
Ymouse_5_xy = 'VK-AA502_shaft_2341-3010_x-y.csv'

Ymouse_1_zy = 'VK-AA498_shaft_2376-3055_z-y.csv'
Ymouse_2_zy = 'VK-AA499_shaft_2065-2655_z-y.csv'
Ymouse_3_zy = 'VK-AA500_shaft_2506-3222_z-y.csv'
Ymouse_4_zy = 'VK-AA501_shaft_2376-3055_z-y.csv'
Ymouse_5_zy = 'VK-AA502_shaft_2341-3010_z-y.csv'

# Young mice: Vessel parallel to central vein on xz-plane
Ymouse_1_xz = 'VK-AA498_shaft_2376-3055_x-z.csv'
Ymouse_2_xz = 'VK-AA499_shaft_2065-2655_x-z.csv'
Ymouse_3_xz = 'VK-AA500_shaft_2506-3222_x-z.csv'
Ymouse_4_xz = 'VK-AA501_shaft_2376-3055_x-z.csv'
Ymouse_5_xz = 'VK-AA502_shaft_2341-3010_x-z.csv'

In [4]:
# Young, blood-loss mice (both x-y and z-y are perpendicular to the central vein)
Bmouse_1_xy = 'VK-AA763_shaft_2142-2754_x-y.csv'
Bmouse_2_xy = 'VK-AA764_shaft_2271-2920_x-y.csv'
Bmouse_3_xy = 'VK-AA765_shaft_2205-2835_x-y.csv'
Bmouse_4_xy = 'VK-AA766_shaft_2096-2695_x-y.csv'
Bmouse_5_xy = 'VK-AA767_shaft_2166-2785_x-y.csv'

Bmouse_1_zy = 'VK-AA763_shaft_2142-2754_z-y.csv'
Bmouse_2_zy = 'VK-AA764_shaft_2271-2920_z-y.csv'
Bmouse_3_zy = 'VK-AA765_shaft_2205-2835_z-y.csv'
Bmouse_4_zy = 'VK-AA766_shaft_2096-2695_z-y.csv'
Bmouse_5_zy = 'VK-AA767_shaft_2166-2785_z-y.csv'

# Young, blood loss mice: Vessel parallel to central vein on xz-plane
Bmouse_1_xz = 'VK-AA763_shaft_2142-2754_x-z.csv'
Bmouse_2_xz = 'VK-AA764_shaft_2271-2920_x-z.csv'
Bmouse_3_xz = 'VK-AA765_shaft_2205-2835_x-z.csv'
Bmouse_4_xz = 'VK-AA766_shaft_2096-2695_x-z.csv'
Bmouse_5_xz = 'VK-AA767_shaft_2166-2785_x-z.csv'

In [5]:
# Old mice (both x-y and z-y are perpendicular to the central vein)
Omouse_1_xy = 'VK-AA758_shaft_2740-3523_x-y.csv'
Omouse_2_xy = 'VK-AA759_shaft_2723-3501_x-y.csv'
Omouse_3_xy = 'VK-AA760_shaft_2866-3685_x-y.csv'
Omouse_4_xy = 'VK-AA761_shaft_2684-3451_x-y.csv'
Omouse_5_xy = 'VK-AA762_shaft_2905-3735_x-y.csv'

Omouse_1_zy = 'VK-AA758_shaft_2740-3523_z-y.csv'
Omouse_2_zy = 'VK-AA759_shaft_2723-3501_z-y.csv'
Omouse_3_zy = 'VK-AA760_shaft_2866-3685_z-y.csv'
Omouse_4_zy = 'VK-AA761_shaft_2684-3451_z-y.csv'
Omouse_5_zy = 'VK-AA762_shaft_2905-3735_z-y.csv'

# Old mice: Vessel parallel to central vein on xz-plane
Omouse_1_xz = 'VK-AA758_shaft_2740-3523_x-z.csv'
Omouse_2_xz = 'VK-AA759_shaft_2723-3501_x-z.csv'
Omouse_3_xz = 'VK-AA760_shaft_2866-3685_x-z.csv'
Omouse_4_xz = 'VK-AA761_shaft_2684-3451_x-z.csv'
Omouse_5_xz = 'VK-AA762_shaft_2905-3735_x-z.csv'

## Define function to plot the overlaid density distribution

In [6]:
# Define functions to read all five datasets of case and to generate overlaid histograms
def read_area_column(data_xy: np.ndarray):
    area =  np.genfromtxt(data_xy ,skip_header=1, delimiter=',', usecols=(1))
    
    return area

# Function to plot the density histograms all in one plot
def plot_histograms(datasets, bins=50, colors=None, labels=None, title="Histogram", xlabel="diameter [µm]", ylabel="Frequency", range=(10, 50)):
    plt.figure(figsize=(10, 6))
    
    for i, data in enumerate(datasets):        
        # Plot histogram
        plt.hist(data, bins=bins, color=colors[i] if colors else None, label=labels[i] if labels else None, histtype='step', lw=2, density=True, range=range)
        
    # Add labels and title
    plt.xlabel(xlabel, fontsize = 14, fontweight = 'bold')
    plt.ylabel(ylabel, fontsize = 14, fontweight = 'bold')
    plt.title(title, fontsize = 16, fontweight = 'bold')

    # Add legend if labels are provided
    if labels:
        plt.legend()

    # Show the plot
    plt.show()
    
# Function to save the density histograms
def save_density_histograms_to_csv(datasets, bins=50, range=(10, 50), csv_filename="combined_density_histograms.csv"):
    
    # Create an empty list to store histogram dataframes
    dfs = []
    
    for i, data in enumerate(datasets):
        # Compute density histogram values
        values, bin_edges = np.histogram(data, bins=bins, range=range, density=True)
        
        # Create a DataFrame for the current dataset's histogram
        df = pd.DataFrame({'Bin Edges': bin_edges[:-1], f'Density Values_{i}': values})  # Add a column with density values for the current datasets
        
        # Append the DataFrame to the list
        dfs.append(df)
    
    # Merge all the individual dataframes on the 'Bin Edges' column
    merged_df = dfs[0]  # Start with the first DataFrame
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on='Bin Edges', how='outer')
    
    # Save the merged DataFrame to the specified CSV file
    merged_df.to_csv(csv_filename, index=False)

## Young mice dataset
### Vessel perpendicular to central vein

In [7]:
# Load young mice data sets
Ydatasets_xy = [Ymouse_1_xy, Ymouse_2_xy, Ymouse_3_xy, Ymouse_4_xy, Ymouse_5_xy]
Ydatasets_zy = [Ymouse_1_zy, Ymouse_2_zy, Ymouse_3_zy, Ymouse_4_zy, Ymouse_5_zy]
Yall_diameter_perp = list()

for dataset_xy, dataset_zy in zip(Ydatasets_xy,Ydatasets_zy):
    area_xy = read_area_column(dataset_xy)
    area_zy = read_area_column(dataset_zy)
    area = np.concatenate([area_xy, area_zy])
    diameter = 2 * np.sqrt(np.array(area)/np.pi)    
    Yall_diameter_perp.append(diameter)

#plot_histograms(Yall_diameter_perp, bins=100, colors=Ycolors, labels=Ylabels, title='Young Mice (LC) - hip / perpendicular')

### Vessel parallel to central vein

In [8]:
# Load young mice data sets
Ydatasets_xz = [Ymouse_1_xz, Ymouse_2_xz, Ymouse_3_xz, Ymouse_4_xz, Ymouse_5_xz]
Yall_diameter_par = list()

for dataset_xz in Ydatasets_xz:
    area_xz = read_area_column(dataset_xz)
    diameter = 2 * np.sqrt(np.array(area_xz)/np.pi)
    Yall_diameter_par.append(diameter)

#plot_histograms(Yall_diameter_par, bins=100, colors=Ycolors, labels=Ylabels, title='Young Mice (LC) - hip / parallel', range=(15,50))

## Young, blood loss mice dataset
### Vessel perpendicular to central vein

In [9]:
# Load young, blood loss mice data sets
Bdatasets_xy = [Bmouse_1_xy, Bmouse_2_xy, Bmouse_3_xy, Bmouse_4_xy, Bmouse_5_xy]
Bdatasets_zy = [Bmouse_1_zy, Bmouse_2_zy, Bmouse_3_zy, Bmouse_4_zy, Bmouse_5_zy]

Ball_diameter_perp = list()

for dataset_xy, dataset_zy in zip(Bdatasets_xy,Bdatasets_zy):
    area_xy = read_area_column(dataset_xy)
    area_zy = read_area_column(dataset_zy)
    area = np.concatenate([area_xy, area_zy])
    diameter = 2 * np.sqrt(np.array(area)/np.pi)    
    Ball_diameter_perp.append(diameter)

#plot_histograms(Ball_diameter_perp, bins=100, colors=Bcolors, labels=Blabels, title='Young, blood loss Mice (LC) - hip / perpendicular')

### Vessel parallel to central vein

In [10]:
# Load young, blood loss mice data sets
Bdatasets_xz = [Bmouse_1_xz, Bmouse_2_xz, Bmouse_3_xz, Bmouse_4_xz, Bmouse_5_xz]
Ball_diameter_par = list()

for dataset_xz in Bdatasets_xz:
    area_xz = read_area_column(dataset_xz)
    diameter = 2 * np.sqrt(np.array(area_xz)/np.pi)
    Ball_diameter_par.append(diameter)

#plot_histograms(Ball_diameter_par, bins=100, colors=Bcolors, labels=Blabels, title='Young, blood loss Mice (LC) - hip / parallel', range=(15,50))

## Aged (old) mice dataset
### Vessel perpendicular to central vein

In [11]:
# Load young, blood loss mice data sets
Odatasets_xy = [Omouse_1_xy, Omouse_2_xy, Omouse_3_xy, Omouse_4_xy, Omouse_5_xy]
Odatasets_zy = [Omouse_1_zy, Omouse_2_zy, Omouse_3_zy, Omouse_4_zy, Omouse_5_zy]

Oall_diameter_perp = list()

for dataset_xy, dataset_zy in zip(Odatasets_xy,Odatasets_zy):
    area_xy = read_area_column(dataset_xy)
    area_zy = read_area_column(dataset_zy)
    area = np.concatenate([area_xy, area_zy])
    diameter = 2 * np.sqrt(np.array(area)/np.pi)    
    Oall_diameter_perp.append(diameter)

#plot_histograms(Oall_diameter_perp, bins=100, colors=Ocolors, labels=Olabels, title='Aged mice (LC) - knee / perpendicular')

### Vessel parallel to central vein

In [12]:
# Load young, blood loss mice data sets
Odatasets_xz = [Omouse_1_xz, Omouse_2_xz, Omouse_3_xz, Omouse_4_xz, Omouse_5_xz]
Oall_diameter_par = list()

for dataset_xz in Odatasets_xz:
    area_xz = read_area_column(dataset_xz)
    diameter = 2 * np.sqrt(np.array(area_xz)/np.pi)
    Oall_diameter_par.append(diameter)

#plot_histograms(Oall_diameter_par, bins=100, colors=Ocolors, labels=Olabels, title='Aged mice (LC) - knee / parallel', range=(15,50))

## Save histograms (density)

In [13]:
save_density_histograms_to_csv(Ball_diameter_par, csv_filename="Blood-let_parallel (mid-knee2).csv")
save_density_histograms_to_csv(Ball_diameter_perp, csv_filename="Blood-let_perpendicular (mid-knee2).csv")
save_density_histograms_to_csv(Yall_diameter_par, csv_filename="Young_parallel (mid-knee2).csv")
save_density_histograms_to_csv(Yall_diameter_perp, csv_filename="Young_perpendicular (mid-knee2).csv")
save_density_histograms_to_csv(Oall_diameter_par, csv_filename="Aged_parallel (mid-knee2).csv")
save_density_histograms_to_csv(Oall_diameter_perp, csv_filename="Aged_perpendicular (mid-knee2).csv")